In [ ]:
from pathlib import Path
import importlib
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import natural_reward_seeking.models.policy as policy_module
policy_module = importlib.reload(policy_module)
get_stop_token_ids = policy_module.get_stop_token_ids
parse_generated_output = policy_module.parse_generated_output
render_messages_to_text = policy_module.render_messages_to_text
trim_trailing_stop_ids = policy_module.trim_trailing_stop_ids

model_id = "allenai/Olmo-3-7B-Think-SFT"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
model = model.to(device).eval()

messages = [{"role": "user", "content": "Explain in one paragraph why the sky appears blue."}]
prompt_text = render_messages_to_text(
    tokenizer,
    messages,
    enable_thinking=True,
    add_generation_prompt=True,
)
encoded = tokenizer(prompt_text, return_tensors="pt")
model_device = next(model.parameters()).device
encoded = {key: value.to(model_device) for key, value in encoded.items()}
stop_ids = get_stop_token_ids(tokenizer)
outputs = model.generate(
    **encoded,
    max_new_tokens=256,
    do_sample=False,
    eos_token_id=(stop_ids or None),
    pad_token_id=tokenizer.pad_token_id,
    use_cache=True,
)
generated_ids = outputs[0, encoded["input_ids"].shape[1]:].tolist()
trimmed_ids, removed_terminal_tokens = trim_trailing_stop_ids(generated_ids, stop_ids)
decoded = tokenizer.decode(trimmed_ids, skip_special_tokens=False)
parsed = parse_generated_output(decoded)
print("RAW OUTPUT:\n", parsed["raw_output"])
print("\nREASONING TRACE:\n", parsed["reasoning_trace"])
print("\nFINAL ANSWER:\n", parsed["answer"])
print("\nREMOVED TERMINAL TOKENS:\n", removed_terminal_tokens)


In [ ]:
from pathlib import Path
import sys
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

reward_model_id = "Skywork/Skywork-Reward-V2-Llama-3.1-8B-40M"
reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_id, use_fast=True)
if reward_tokenizer.pad_token is None:
    reward_tokenizer.pad_token = reward_tokenizer.eos_token or reward_tokenizer.unk_token
reward_model = AutoModelForSequenceClassification.from_pretrained(
    reward_model_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
reward_model = reward_model.to("cuda" if torch.cuda.is_available() else "cpu").eval()

messages = [
    {"role": "user", "content": "Explain in one paragraph why the sky appears blue."},
    {
        "role": "assistant",
        "content": "The sky looks blue because tiny molecules in the atmosphere scatter short-wavelength blue light more strongly than longer-wavelength red light, so blue light reaches our eyes from many directions.",
    },
]
if hasattr(reward_tokenizer, "apply_chat_template"):
    rendered = reward_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
else:
    rendered = "\n".join(f"{message['role'].upper()}: {message['content']}" for message in messages)

encoded = reward_tokenizer(
    rendered,
    return_tensors="pt",
    truncation=True,
    padding=True,
)
reward_device = next(reward_model.parameters()).device
encoded = {key: value.to(reward_device) for key, value in encoded.items()}
with torch.inference_mode():
    outputs = reward_model(**encoded)
score = float(outputs.logits[0, 0].detach().float().cpu().item())
print("RENDERED INPUT:\n", rendered)
print("\nREWARD SCORE:\n", score)
